<a href="https://colab.research.google.com/github/dechl-98/ChavezDavid2534532021/blob/main/ClaveG/Ejercicio3_Clusterizaci%C3%B3n_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Caso 3 Agrupación de datos ClaveG_agrupacion
#Importando las librerías que se necesitarán para trabajar y presentar los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [4]:
#Mandamos a llamar la data desde el repositorio e imprimimos los resultados actuales
url = "https://raw.githubusercontent.com/dechl-98/ChavezDavid2534532021/refs/heads/main/ClaveG/clave_G_agrupacion.csv"
df = pd.read_csv(url)
print(df.head())
print(df.shape)
print(df.info())

  registro_id  edad  ingresos  frecuencia_uso  gasto_promedio  satisfaccion  \
0     G-R0007    33       631            3.98           15.70          6.07   
1     G-R0123    44       995            6.13           75.82          7.87   
2     G-R0009    28       509            1.30           30.45          5.97   
3     G-R0228    46       769            2.94           69.77          5.28   
4     G-R0106    37      1069            5.56           83.50          8.27   

   reclamos  antiguedad_meses  
0         3                 1  
1         2                32  
2         5                 1  
3        12                21  
4         1                 8  
(254, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   registro_id       254 non-null    object 
 1   edad              254 non-null    int64  
 2   ingresos          254 non-n

contemplamos un total de 254 registros, 8 columnas

In [8]:
#Revisión de valores nulos y duplicados
df.isnull().sum()
df.duplicated().sum()

np.int64(0)

no se reportan valores nulos nii duplicados

In [9]:
#Verificamos valores numericos y revisamos variables atípicas
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

for columna in numerical_cols:
    media = df[columna].mean()
    desviacion = df[columna].std()

    limite_inferior = media - 2 * desviacion
    limite_superior = media + 2 * desviacion

    atipicos = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]
    print(f"Columna: {columna}, Límite Inferior: {limite_inferior:.2f}, Límite Superior: {limite_superior:.2f}, Cantidad de Atípicos: {len(atipicos)}")

Columna: edad, Límite Inferior: 20.11, Límite Superior: 59.70, Cantidad de Atípicos: 9
Columna: ingresos, Límite Inferior: 187.35, Límite Superior: 2006.11, Cantidad de Atípicos: 2
Columna: frecuencia_uso, Límite Inferior: -0.31, Límite Superior: 11.37, Cantidad de Atípicos: 6
Columna: gasto_promedio, Límite Inferior: -13.10, Límite Superior: 184.37, Cantidad de Atípicos: 4
Columna: satisfaccion, Límite Inferior: 2.97, Límite Superior: 11.04, Cantidad de Atípicos: 4
Columna: reclamos, Límite Inferior: -1.93, Límite Superior: 6.81, Cantidad de Atípicos: 9
Columna: antiguedad_meses, Límite Inferior: -8.04, Límite Superior: 46.44, Cantidad de Atípicos: 11


The previous error indicated that `X_scaled_df` contains NaN values, likely due to the `satisfaccion` column having one missing entry as seen in `df.info()`. K-Means cannot handle NaNs. We will impute this missing value using the mean of the `satisfaccion` column before scaling.

In [13]:
# Impute missing values in 'satisfaccion' column with the mean
if 'satisfaccion' in df.columns and df['satisfaccion'].isnull().any():
    df['satisfaccion'] = df['satisfaccion'].fillna(df['satisfaccion'].mean())
    print("Missing values in 'satisfaccion' filled with mean.")

numerical_cols_for_clustering = df.select_dtypes(include=np.number).columns.tolist()
X = df[numerical_cols_for_clustering]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert scaled data back to a DataFrame for easier interpretation of centroids later
X_scaled_df = pd.DataFrame(X_scaled, columns=numerical_cols_for_clustering)

print("\nCheck for NaNs after imputation and scaling:")
print(X_scaled_df.isnull().sum())

Missing values in 'satisfaccion' filled with mean.

Check for NaNs after imputation and scaling:
edad                0
ingresos            0
frecuencia_uso      0
gasto_promedio      0
satisfaccion        0
reclamos            0
antiguedad_meses    0
dtype: int64


In [14]:
modelo_k4 = KMeans(n_clusters=4, random_state=0, n_init=10)
df["clusterk4"] = modelo_k4.fit_predict(X_scaled_df)

# To get centroids in original scale, inverse transform the scaled centroids
centroides_k4 = pd.DataFrame(scaler.inverse_transform(modelo_k4.cluster_centers_), columns=numerical_cols_for_clustering)
print("\nCentroides de los clusters (valores originales):")
print(centroides_k4)
print("\nConteo de registros por cluster:")
print(df["clusterk4"].value_counts())


Centroides de los clusters (valores originales):
        edad     ingresos  frecuencia_uso  gasto_promedio  satisfaccion  \
0  28.515625   629.578125        2.276563       38.650938      6.060469   
1  51.366667  1731.666667        9.075500      157.002333      8.939891   
2  43.310345   921.155172        4.097586       60.181897      4.365345   
3  37.736111  1124.291667        6.634861       88.429167      8.353056   

   reclamos  antiguedad_meses  
0  3.218750          8.328125  
1  0.616667         38.016667  
2  5.362069         12.672414  
3  0.902778         18.444444  

Conteo de registros por cluster:
clusterk4
3    72
0    64
1    60
2    58
Name: count, dtype: int64
